In [14]:
import hashlib, os
import polars as pl
import duckdb

DATA_PATH = "../data/raw/data.csv"  
assert os.path.exists(DATA_PATH), f"File tidak ditemukan: {DATA_PATH}"

size_mb = os.path.getsize(DATA_PATH) / 1024**2
print(f"Ukuran file: {size_mb:,.2f} MB")

h = hashlib.sha256()
with open(DATA_PATH, "rb") as f:
    for block in iter(lambda: f.read(1024*1024), b""):
        h.update(block)
print("SHA-256:", h.hexdigest())   

Ukuran file: 703.73 MB
SHA-256: cdde95f6de7c2f6a2af5d52df532f934f5ddb7381a59b3d1f99e7a475deadcbb


In [6]:
lf = pl.scan_csv(DATA_PATH, infer_schema_length=0)
cols = lf.collect_schema().names()
print(len(cols), "kolom:", cols)

11 kolom: ['id', 'source', 'title', 'image', 'url', 'content', 'date', 'embedding', 'created_at', 'updated_at', 'summary']


In [7]:
n_rows = lf.select(pl.len()).collect().item()
print(f"Jumlah baris: {n_rows:,}")

Jumlah baris: 32,735


In [8]:
check_cols = cols
nulls = lf.select(
    [pl.col(c).is_null().sum().alias(c) for c in check_cols]
    + [(pl.col(c).str.strip_chars() == "").sum().alias(f"{c}__kosong") for c in check_cols]
).collect()
pl.Config.set_tbl_cols(30)
nulls

id,source,title,image,url,content,date,embedding,created_at,updated_at,summary,id__kosong,source__kosong,title__kosong,image__kosong,url__kosong,content__kosong,date__kosong,embedding__kosong,created_at__kosong,updated_at__kosong,summary__kosong
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,10,0,0,0,160,0,441,0,0,0,0,0


In [9]:
for c in [c for c in ("id", "url", "title") if c in cols]:
    n_unique = lf.select(pl.col(c).n_unique()).collect().item()
    print(f"{c}: {n_rows - n_unique:,} nilai duplikat dari {n_rows:,} baris")

id: 0 nilai duplikat dari 32,735 baris
url: 0 nilai duplikat dari 32,735 baris
title: 84 nilai duplikat dari 32,735 baris


In [10]:
if "created_at" in cols:
    dt = lf.select(
        pl.col("created_at").str.to_datetime(strict=False).alias("created_at")
    )
    print(dt.select(
        pl.col("created_at").min().alias("min"),
        pl.col("created_at").max().alias("max"),
        pl.col("created_at").is_null().sum().alias("gagal_parse"),
    ).collect())
    display(
        dt.group_by(pl.col("created_at").dt.year().alias("tahun"))
          .agg(pl.len().alias("n")).sort("tahun").collect()
    )

shape: (1, 3)
┌────────────────────────────┬────────────────────────────┬─────────────┐
│ min                        ┆ max                        ┆ gagal_parse │
│ ---                        ┆ ---                        ┆ ---         │
│ datetime[μs]               ┆ datetime[μs]               ┆ u32         │
╞════════════════════════════╪════════════════════════════╪═════════════╡
│ 2023-03-03 16:08:03.144472 ┆ 2023-04-11 20:30:13.315332 ┆ 0           │
└────────────────────────────┴────────────────────────────┴─────────────┘


tahun,n
i32,u32
2023,32735


In [11]:
if "url" in cols:
    per_site = (
        lf.select(pl.col("url").str.extract(r"https?://([^/]+)", 1).alias("domain"))
          .group_by("domain").agg(pl.len().alias("n"))
          .sort("n", descending=True).collect()
    )
    print(per_site)

shape: (11, 2)
┌─────────────────────────┬──────┐
│ domain                  ┆ n    │
│ ---                     ┆ ---  │
│ str                     ┆ u32  │
╞═════════════════════════╪══════╡
│ kumparan.com            ┆ 8579 │
│ www.cnbcindonesia.com   ┆ 4691 │
│ www.suara.com           ┆ 4004 │
│ www.cnnindonesia.com    ┆ 3738 │
│ economy.okezone.com     ┆ 2537 │
│ …                       ┆ …    │
│ nasional.okezone.com    ┆ 1878 │
│ news.okezone.com        ┆ 1862 │
│ bisnis.tempo.co         ┆ 1467 │
│ www.jawapos.com         ┆ 1015 │
│ megapolitan.okezone.com ┆ 789  │
└─────────────────────────┴──────┘


In [12]:
if "content" in cols:
    stats = lf.select(pl.col("content").str.len_chars().alias("len")).select(
        pl.col("len").min().alias("min"),
        pl.col("len").quantile(0.5).alias("median"),
        pl.col("len").quantile(0.95).alias("p95"),
        pl.col("len").max().alias("max"),
        (pl.col("len") < 100).sum().alias("kurang_dari_100_karakter"),
    ).collect()
    print(stats)

shape: (1, 5)
┌─────┬────────┬────────┬───────┬──────────────────────────┐
│ min ┆ median ┆ p95    ┆ max   ┆ kurang_dari_100_karakter │
│ --- ┆ ---    ┆ ---    ┆ ---   ┆ ---                      │
│ u32 ┆ f64    ┆ f64    ┆ u32   ┆ u32                      │
╞═════╪════════╪════════╪═══════╪══════════════════════════╡
│ 0   ┆ 2070.0 ┆ 4394.0 ┆ 13833 ┆ 443                      │
└─────┴────────┴────────┴───────┴──────────────────────────┘


In [13]:
select_cols = ", ".join(f'"{c}"' for c in cols if c != "embedding")
con = duckdb.connect()
con.sql(f'''
    SUMMARIZE
    SELECT {select_cols}
    FROM read_csv('{DATA_PATH}', all_varchar=true, header=true)
''').show(max_width=250)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────────┬──────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────┬───────────────┬────────┬────────┬────────┬────────┬────────┬───────┬─────────────────┐
│ column_name │ column_type │                               min                                │                               max                                │ approx_unique │  avg   │  std   │  q25   │  q50   │  q75   │ count │ null_percentage │
│   varchar   │   varchar   │                             varchar                              │                             varchar                              │     int64     │ "null" │ "null" │ "null" │ "null" │ "null" │ int64 │  decimal(9,2)   │
├─────────────┼─────────────┼──────────────────────────────────────────────────────────────────┼──────────────────────────────────────────────────────────────────┼───────────────┼────────┼────────┼────────┼────────┼────────┼───────┼───────────────

## Catatan Milestone 1
- Jumlah baris: 32,735
- Rentang waktu `created_at`: 2023-03-03 16:08:03.144472 ┆ 2023-04-11 20:30:13.315332 ┆ 0  
- Jumlah situs / domain unik: 11 domain
- Kolom dengan banyak null/kosong: Kolom summary memiliki 160.044 nilai string kosong, sedangkan kolom utama seperti id, url, title, dan content sepenuhnya terisi (0 null / 0 kosong).
- Jumlah duplikat (`id`, `url`):
  - id: 0 nilai duplikat dari 32,735 baris
  - url: 0 nilai duplikat dari 32,735 baris
  - title: 84 nilai duplikat dari 32,735 baris